In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/thule/miniconda3/envs/open-clip-training/lib/python311.zip', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/lib-dynload', '', '/home/thule/miniconda3/envs/open-clip-training/lib/python3.11/site-packages', '../']


In [2]:
# from networks.resnet_big import ConResNet
# from torchinfo import summary
from thop import profile
import torch
import torch.nn as nn 

In [3]:
import torch
import torch.nn as nn
from thop import profile, clever_format
 
 
def count_macs_and_params(model: nn.Module, input_shape: tuple, device: str = "cpu"):
    model = model.to(device).eval()
    dummy_input = torch.randn(1, *input_shape).to(device)
 
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    flops = macs * 2  # quy đổi MACs -> FLOPs (1 MAC = 2 FLOPs)
    macs_str, params_str, flops_str = clever_format([macs, params, flops], "%.3f")
 
    return {
        "macs": int(macs),
        "flops": int(flops),
        "params": int(params),
        "macs_readable": macs_str,
        "flops_readable": flops_str,
        "params_readable": params_str,
    }

In [4]:
import time
import torch

def process_frame(model, frame_data, device):
    start_time = time.time()
    
    model.eval()
    frame_data = frame_data.to(device)
    
    with torch.no_grad(): 
        model(frame_data)
        
    end_time = time.time()
    processing_time = (end_time - start_time) * 1000
    return processing_time


In [11]:
from networks.conv_retnet import ConvRetNet


cnn_1d_retnet_model = ConvRetNet(
                num_features=9,
                d_model=128,
                out_channels_payload=128,
                window_size=32,
                num_layers=2,
                num_heads=4,
                ffn_dim=128*2,
                num_classes=10,
                dropout=0.1,
            )

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cnn_1d_retnet_model = cnn_1d_retnet_model.to(device)

In [16]:
print(count_macs_and_params(cnn_1d_retnet_model, input_shape=(32, 9), device='cuda'))

{'macs': 11282688, 'flops': 22565376, 'params': 400266, 'macs_readable': '11.283M', 'flops_readable': '22.565M', 'params_readable': '400.266K'}


In [18]:
import time
import numpy as np
import torch


def benchmark_inference(model, frame_data, device, n_warmup=10, n_runs=100):
    model = model.to(device).eval()
    frame_data = frame_data.to(device)

    is_cuda = (device == "cuda" or (isinstance(device, torch.device) and device.type == "cuda"))

    # ── Bước 1: WARMUP - chạy vài lần đầu KHÔNG TÍNH ──
    with torch.no_grad():
        for _ in range(n_warmup):
            model(frame_data)
    if is_cuda:
        torch.cuda.synchronize()  # đợi warmup chạy XONG THẬT trước khi bắt đầu đo

    # ── Bước 2: ĐO NHIỀU LẦN, đồng bộ hóa GPU trước/sau MỖI lần ──
    times_ms = []
    with torch.no_grad():
        for _ in range(n_runs):
            if is_cuda:
                torch.cuda.synchronize()  # đảm bảo GPU đã rảnh TRƯỚC khi bắt đầu đếm giờ

            start_time = time.perf_counter()  # chính xác hơn time.time() cho benchmark
            model(frame_data)
            if is_cuda:
                torch.cuda.synchronize()  # đợi GPU tính XONG THẬT rồi mới dừng đồng hồ
            end_time = time.perf_counter()

            times_ms.append((end_time - start_time) * 1000)

    times_ms = np.array(times_ms)
    return {
        "mean_ms": float(times_ms.mean()),
        "std_ms": float(times_ms.std()),
        "min_ms": float(times_ms.min()),
        "max_ms": float(times_ms.max()),
        "all_times_ms": times_ms.tolist(),
    }

In [19]:
model = cnn_1d_retnet_model
frame_data = torch.rand(1, 32, 9)

result = benchmark_inference(model, frame_data, device, n_warmup=10, n_runs=100)
print(f"Device: {device}")
print(f"Mean processing time: {result['mean_ms']:.3f} ms (± {result['std_ms']:.3f} ms)")
print(f"Min/Max: {result['min_ms']:.3f} ms / {result['max_ms']:.3f} ms")

Device: cuda
Mean processing time: 3.240 ms (± 0.310 ms)
Min/Max: 3.000 ms / 6.240 ms


In [ ]:
# ============================================================
# Inference-time measurement for the proposed IDS (Jupyter cell)
# ============================================================
import time
import numpy as np
import torch

# ---------------- 1. Configuration (edit to match your setup) ----------------
WINDOW_SIZE = 32          # T: number of CAN frames per window
NUM_FEATURES = 9          # F: features per frame (e.g. CAN ID + 8 payload bytes)
BATCH_SIZES = [1, 128]    # 1 = single-window latency, 128 = batched throughput
N_WARMUP = 50             # warm-up runs (not timed)
N_RUNS = 1000             # timed runs
DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
CHECKPOINT = None         # e.g. "checkpoints/last.pth", or None to skip loading
torch.set_num_threads(1)  # single CPU thread, closer to an embedded ECU setting

# ---------------- 2. Build the model (replace with your own) ----------------
from model import ConvRetNet               # <-- adjust the import to your project
model = ConvRetNet()                       # <-- pass the same arguments as in training
if CHECKPOINT is not None:
    state = torch.load(CHECKPOINT, map_location="cpu")
    model.load_state_dict(state.get("model", state))
model.eval()

# ---------------- 3. Model size and complexity ----------------
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters : {n_params:,}")
print(f"Model size : {n_params * 4 / 1024**2:.3f} MB (float32)")

try:  # optional: pip install thop
    from thop import profile
    dummy = torch.randn(1, WINDOW_SIZE, NUM_FEATURES)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    print(f"MACs/window: {macs / 1e6:.3f} M")
except Exception as e:
    print(f"MACs not computed ({e})")

# ---------------- 4. Latency measurement ----------------
def measure(model, device, batch_size):
    m = model.to(device)
    x = torch.randn(batch_size, WINDOW_SIZE, NUM_FEATURES, device=device)
    sync = torch.cuda.synchronize if device == "cuda" else (lambda: None)
    times = []
    with torch.inference_mode():
        for _ in range(N_WARMUP):          # warm-up: caches, kernel selection
            m(x)
        sync()
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            m(x)
            sync()                          # wait for GPU kernels to finish
            times.append((time.perf_counter() - t0) * 1000)  # ms
    t = np.array(times)
    return {
        "mean": t.mean(), "std": t.std(), "median": np.median(t),
        "p95": np.percentile(t, 95),
        "per_window": t.mean() / batch_size,
        "frames_per_s": batch_size * WINDOW_SIZE / (t.mean() / 1000),
    }

print(f"\n{'Device':<6} {'Batch':>5} | {'Mean':>8} {'Std':>7} {'Median':>8} "
      f"{'P95':>8} | {'ms/window':>9} {'frames/s':>12}")
print("-" * 82)
for device in DEVICES:
    for bs in BATCH_SIZES:
        r = measure(model, device, bs)
        print(f"{device:<6} {bs:>5} | {r['mean']:8.3f} {r['std']:7.3f} "
              f"{r['median']:8.3f} {r['p95']:8.3f} | {r['per_window']:9.4f} "
              f"{r['frames_per_s']:12,.0f}")

# ---------------- 5. Real-time feasibility check ----------------
FRAME_TIME_MS = 0.25  # approx. duration of one 8-byte CAN frame at 500 kbit/s
window_time = WINDOW_SIZE * FRAME_TIME_MS
latency = measure(model, "cpu", 1)["mean"]
print(f"\nWindow accumulation time (full bus load): {window_time:.2f} ms")
print(f"Single-window CPU latency               : {latency:.3f} ms")
print("Real-time capable" if latency < window_time else "NOT real-time capable")